In [26]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    silhouette_score
)

df = pd.read_csv("battery_observations.csv", index_col=0)
meta = pd.read_csv("battery_metadata.csv", index_col=0)
df2 = df.join(meta[["climate_zone", "chemistry"]])
df2 = df2.sort_values(["battery_id", "cycle_index"])
df2 = df2.reset_index()
meta = meta.reset_index()
g = lambda x: int(x.split('-')[-1])
df2["battery_id"] = df2["battery_id"].apply(g)
meta["battery_id"] = meta["battery_id"].apply(g)
battery_ids = list(set(df2["battery_id"]))
materials = list(set(meta["chemistry"]))
climate_zones = list(set(meta["climate_zone"]))
labels = list(set(df2["degradation_status"]))

train_labels = pd.read_csv("train_batteries.csv")["battery_id"].apply(g)
val_labels = pd.read_csv("validation_batteries.csv")["battery_id"].apply(g)

train = df2[df2["battery_id"].isin(train_labels)]
val = df2[df2["battery_id"].isin(val_labels)]

In [27]:

cycle_features = [
    "soc_start_pct",
    "soc_end_pct",
    "depth_of_discharge_pct",
    "high_soc_exposure_fraction",
    "low_soc_exposure_fraction",
    "charge_c_rate",
    "discharge_c_rate",
    "charge_current_a",
    "discharge_current_a",
    "charge_duration_min",
    "discharge_duration_min",
    "energy_discharged_kwh",
    "ambient_temp_c",
    #"pack_temp_c",
    "cooling_active",
    "pack_voltage_mean_v"
]

extended_features = cycle_features + [
    "battery_age_days",
    "time_since_previous_cycle_hours",
    "equivalent_full_cycles",
    "cumulative_high_temp_hours",
    "pack_voltage_charge_v",
    "pack_voltage_discharge_v"
]

features_A = [
    "battery_age_days",
    "equivalent_full_cycles",
]

features_B = features_A + [
    "internal_resistance_mOhm",
]

features_C = features_B + [
    "cumulative_high_temp_hours",
]


In [28]:
import pandas as pd
import numpy as np

X_train_raw = train[cycle_features]
X_val_raw = val[cycle_features]

# 1. Total NaNs per feature
print("Train NaNs per column:")
print(X_train_raw.isna().sum().sort_values(ascending=False))

print("\nValidation NaNs per column:")
print(X_val_raw.isna().sum().sort_values(ascending=False))

# 2. Overall counts
print("\nTrain rows with any NaN:", X_train_raw.isna().any(axis=1).sum())
print("Train total NaN cells:", X_train_raw.isna().sum().sum())

print("\nValidation rows with any NaN:", X_val_raw.isna().any(axis=1).sum())
print("Validation total NaN cells:", X_val_raw.isna().sum().sum())

# 3. Show the offending rows
train_nan_rows = X_train_raw[X_train_raw.isna().any(axis=1)]
val_nan_rows = X_val_raw[X_val_raw.isna().any(axis=1)]

print("\nTrain rows containing NaNs:")
print(train_nan_rows.head(20))

print("\nValidation rows containing NaNs:")
print(val_nan_rows.head(20))

Train NaNs per column:
soc_start_pct                 0
soc_end_pct                   0
depth_of_discharge_pct        0
high_soc_exposure_fraction    0
low_soc_exposure_fraction     0
charge_c_rate                 0
discharge_c_rate              0
charge_current_a              0
discharge_current_a           0
charge_duration_min           0
discharge_duration_min        0
energy_discharged_kwh         0
ambient_temp_c                0
cooling_active                0
pack_voltage_mean_v           0
dtype: int64

Validation NaNs per column:
soc_start_pct                 0
soc_end_pct                   0
depth_of_discharge_pct        0
high_soc_exposure_fraction    0
low_soc_exposure_fraction     0
charge_c_rate                 0
discharge_c_rate              0
charge_current_a              0
discharge_current_a           0
charge_duration_min           0
discharge_duration_min        0
energy_discharged_kwh         0
ambient_temp_c                0
cooling_active                0
pack_vo

In [ ]:


def run_kmeans(k, X_train, X_val, y_train, y_val):

    sample_idx = np.random.choice(
        len(X_train),
        size=2000,
        replace=False
    )

    km = KMeans(n_clusters=k, n_init=25, random_state=42)
    train_clusters = km.fit_predict(X_train) # array of cluster labels
    val_clusters = km.predict(X_val) # array of cluster labels

    # map each cluster to its majority training label
    cluster_to_status = (
        pd.DataFrame({"cluster": train_clusters, "status": y_train})
        .groupby("cluster")["status"]
        .agg(lambda s: s.mode().iloc[0])
        .to_dict()
    ) # looks like {0: 'Severe', 1: 'Normal'} (example)
    val_pred = np.array([cluster_to_status[c] for c in val_clusters])
    metrics = {
        "k": k,
        "inertia": km.inertia_,
        "silhouette": silhouette_score(X_train[sample_idx],
                                       train_clusters[sample_idx]),
        "ARI": adjusted_rand_score(y_val, val_clusters),
        "NMI": normalized_mutual_info_score(y_val, val_clusters),
        "accuracy": accuracy_score(y_val, val_pred),
        "macro_F1": f1_score(y_val, val_pred, labels=class_names,
                             average="macro", zero_division=0),
        "n_classes_predicted": len(np.unique(val_pred)),
    }
    details = {
        "model": km,
        "train_clusters": train_clusters,
        "val_clusters": val_clusters,
        "cluster_to_status": cluster_to_status,
        "val_pred": val_pred,
    }

    return metrics, details, val_pred # for potential confusion matrix later

def predict_from_clusters(val_clusters, mapping):
    return np.array([
        mapping[cluster]
        for cluster in val_clusters
    ])

### K-Means, varying over k, with PCA for different amounts of components

In [39]:

use_features = features_A # change feature set
kmeans_cluster_max = 8
kmeans_cluster_candidates = range(2, kmeans_cluster_max + 1) # k does not need to equal the number of degradation classes.
optimize_metric = "ARI" # out of ARI, silhouette, NMI, accuracy, macro_F1

# 1. Fit preprocessing BEFORE PCA on training data
base_preprocess = Pipeline([
    ("scaler", StandardScaler()),
    ("variance", VarianceThreshold(threshold=0.0))
])

X_train_prepared = base_preprocess.fit_transform(
    train[use_features]
)

X_val_prepared = base_preprocess.transform(
    val[use_features]
)

y_train = train["degradation_status"].to_numpy()
y_val = val["degradation_status"].to_numpy()

class_names = ["Normal", "Moderate", "Severe", "Critical"]

max_components = min(X_train_prepared.shape)

# You can reduce this if the full search is slow.
pca_component_candidates = range(1, max_components + 1)

# Use sampling for silhouette if the dataset is large.
silhouette_sample_size = 10_000

metrics_df = []
val_pred_structs = np.empty((max_components, kmeans_cluster_max - 1), dtype=object)

k_idx_mapping = {x:x-2 for x in kmeans_cluster_candidates}
pca_idx_mapping = {x:x-1 for x in pca_component_candidates}

for i,n_components in enumerate(pca_component_candidates):

    pca = PCA(
        n_components=n_components,
        svd_solver="full",
        whiten=False,
        random_state=42
    )

    X_train_pca = pca.fit_transform(X_train_prepared)
    X_val_pca = pca.transform(X_val_prepared)

    explained_variance = pca.explained_variance_ratio_.sum()

    for j,k in enumerate(kmeans_cluster_candidates):

        metrics, details, val_pred = run_kmeans(k, X_train_prepared, X_val_prepared, y_train, y_val)
        metrics["n_pca"] = n_components
        metrics_df.append(metrics)
        val_pred_structs[i,j] = val_pred

metrics_df = pd.DataFrame(metrics_df).sort_values(optimize_metric, ascending=False)
best_k = metrics_df["k"].values[0]
best_pca = metrics_df["n_pca"].values[0]

print(pca_idx_mapping[best_pca], k_idx_mapping[best_k])

val_predictions_best = val_pred_structs[pca_idx_mapping[best_pca], k_idx_mapping[best_k]]

confusion_df = pd.DataFrame(
    confusion_matrix(
        y_val,
        val_predictions_best,
        labels=class_names
    ),
    index=[f"Actual {label}" for label in class_names],
    columns=[f"Predicted {label}" for label in class_names]
    )

print("\nValidation confusion matrix:")
print(confusion_df)


print("\nValidation classification report:")
print(classification_report(
    y_val,
    val_predictions_best,
    labels=class_names,
    zero_division=0
))


0 2

Validation confusion matrix:
                 Predicted Normal  Predicted Moderate  Predicted Severe  \
Actual Normal                7338                1300                 0   
Actual Moderate              1027                7076               444   
Actual Severe                  78                5798              1898   
Actual Critical                 0                  19                 8   

                 Predicted Critical  
Actual Normal                     0  
Actual Moderate                   0  
Actual Severe                     0  
Actual Critical                   0  

Validation classification report:
              precision    recall  f1-score   support

      Normal       0.87      0.85      0.86      8638
    Moderate       0.50      0.83      0.62      8547
      Severe       0.81      0.24      0.37      7774
    Critical       0.00      0.00      0.00        27

    accuracy                           0.65     24986
   macro avg       0.54      0.48      

### K-Means, k varying

In [ ]:

use_features = features_A # change feature set
kmeans_cluster_max = 8
kmeans_cluster_candidates = range(2, kmeans_cluster_max + 1) # k does not need to equal the number of degradation classes.
optimize_metric = "ARI" # out of ARI, silhouette, NMI, accuracy, macro_F1

# 1. Fit preprocessing BEFORE PCA on training data
base_preprocess = Pipeline([
    ("scaler", StandardScaler()),
    ("variance", VarianceThreshold(threshold=0.0))
])

X_train_prepared = base_preprocess.fit_transform(
    train[use_features]
)

X_val_prepared = base_preprocess.transform(
    val[use_features]
)

y_train = train["degradation_status"].to_numpy()
y_val = val["degradation_status"].to_numpy()

class_names = ["Normal", "Moderate", "Severe", "Critical"]

# Use sampling for silhouette if the dataset is large.
silhouette_sample_size = 10_000

metrics_df = []
val_pred_structs = np.empty(kmeans_cluster_max - 1, dtype=object)

k_idx_mapping = {x:x-2 for x in kmeans_cluster_candidates}

explained_variance = pca.explained_variance_ratio_.sum()

for j,k in enumerate(kmeans_cluster_candidates):

    metrics, details, val_pred = run_kmeans(k, X_train_prepared, X_val_prepared, y_train, y_val)
    metrics_df.append(metrics)
    val_pred_structs[j] = val_pred

metrics_df = pd.DataFrame(metrics_df).sort_values(optimize_metric, ascending=False)
best_k = metrics_df["k"].values[0]

val_predictions_best = val_pred_structs[k_idx_mapping[best_k]]

confusion_df = pd.DataFrame(
    confusion_matrix(
        y_val,
        val_predictions_best,
        labels=class_names
    ),
    index=[f"Actual {label}" for label in class_names],
    columns=[f"Predicted {label}" for label in class_names]
    )

print("\nValidation confusion matrix:")
print(confusion_df)


print("\nValidation classification report:")
print(classification_report(
    y_val,
    val_predictions_best,
    labels=class_names,
    zero_division=0
))


0 2

Validation confusion matrix:
                 Predicted Normal  Predicted Moderate  Predicted Severe  \
Actual Normal                7338                1300                 0   
Actual Moderate              1027                7076               444   
Actual Severe                  78                5798              1898   
Actual Critical                 0                  19                 8   

                 Predicted Critical  
Actual Normal                     0  
Actual Moderate                   0  
Actual Severe                     0  
Actual Critical                   0  

Validation classification report:
              precision    recall  f1-score   support

      Normal       0.87      0.85      0.86      8638
    Moderate       0.50      0.83      0.62      8547
      Severe       0.81      0.24      0.37      7774
    Critical       0.00      0.00      0.00        27

    accuracy                           0.65     24986
   macro avg       0.54      0.48      